# Feature Engineering
This notebook loads the data, performs feature engineering, and adds the required columns. It also ensures all string data and column names are capitalized.

In [6]:
import pandas as pd
import numpy as np

# Load datasets
avg_path = '../data/implemented_data/avg_years_of_education_p.csv'
ill_path = '../data/implemented_data/illeteracy_p.csv'
lit_path = '../data/implemented_data/literacy_p.csv'

df_avg = pd.read_csv(avg_path)
df_ill = pd.read_csv(ill_path)
df_lit = pd.read_csv(lit_path)

# Ensure everything is in capital in all dataframes
for df in [df_avg, df_ill, df_lit]:
    df.columns = df.columns.str.upper()
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.upper()


/var/folders/_x/xjprn7fn01v4sclwmkr7t81c0000gn/T/ipykernel_15693/1746468436.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
/var/folders/_x/xjprn7fn01v4sclwmkr7t81c0000gn/T/ipykernel_15693/1746468436.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://

## 1. Average Years of Education
Adding **GDP PER SCHOOLING YEAR**, **EDUCATION INDEX**, **EDUCATION LEVEL CATEGORY**, and **LITERACY%**.

In [7]:
# GDP per Schooling Year
df_avg['GDP PER SCHOOLING YEAR'] = np.where(
    df_avg['AVG_YEARS_OF_EDUCATION'] > 0,
    df_avg['GDP_PCAP'] / df_avg['AVG_YEARS_OF_EDUCATION'],
    np.nan
)

# Education Index
df_avg['EDUCATION INDEX'] = (df_avg['AVG_YEARS_OF_EDUCATION'] / 15 + df_avg['LITERACY_RATE'] / 100) / 2

# Education Level Category
df_avg['EDUCATION LEVEL CATEGORY'] = pd.cut(
    df_avg['AVG_YEARS_OF_EDUCATION'], 
    bins=[-np.inf, 6, 10, np.inf], 
    labels=['LOW', 'MEDIUM', 'HIGH']
)


## 2. Illiteracy
Adding **LITERACY GROWTH RATE**, **ILLITERACY TO LITERACY RATIO**, and **ILLITERACY GROWTH RATE**.

In [8]:
# literacy%
df_ill['ILLITERACY%'] = df_ill['ILLITERACY_RATE'].round(2)
df_avg.head()

# Literacy Growth Rate
df_ill = df_ill.sort_values(by=['COUNTRY', 'YEAR'])
df_ill['LITERACY GROWTH RATE'] = (df_ill.groupby('COUNTRY')['LITERACY_RATE'].pct_change() * 100).fillna(0)

# Illiteracy to Literacy Ratio
df_ill['ILLITERACY TO LITERACY RATIO'] = np.where(
    df_ill['LITERACY_RATE'] > 0, 
    df_ill['ILLITERACY_RATE'] / df_ill['LITERACY_RATE'], 
    np.nan
)


## 3. Literacy
Adding **LITERACY GAP GENDER**, **YOUTH LITERACY AVERAGE**, and **ADULT VS YOUTH LITERACY GAP**.

In [9]:
# literacy gap gender
df_lit['LITERACY GAP GENDER'] = abs(df_lit['YOUTH_LITERACY_M'] - df_lit['YOUTH_LITERACY_F'])

# Youth Literacy Average
df_lit['YOUTH LITERACY AVERAGE'] = (df_lit['YOUTH_LITERACY_M'] + df_lit['YOUTH_LITERACY_F']) / 2

# Adult vs Youth Literacy Gap
df_lit['ADULT VS YOUTH LITERACY GAP'] = df_lit['YOUTH LITERACY AVERAGE'] - df_lit['ADULT_LITERACY']

df_lit.head()


,COUNTRY,CODE,YEAR,ADULT_LITERACY,YOUTH_LITERACY_M,YOUTH_LITERACY_F,REGION,LITERACY GAP GENDER,YOUTH LITERACY AVERAGE,ADULT VS YOUTH LITERACY GAP
0,AFGHANISTAN,AFG,2011,31.00000,62.00000,32.00000,ASIA,30.00000,47.000000,16.000000
1,AFGHANISTAN,AFG,2015,33.75384,57.73505,25.48416,ASIA,32.25089,41.609605,7.855765
2,AFGHANISTAN,AFG,2021,37.00000,71.00000,42.00000,ASIA,29.00000,56.500000,19.500000
3,AFGHANISTAN,AFG,2022,37.00000,83.40000,44.17171,ASIA,39.22829,63.785855,26.785855
4,ALBANIA,ALB,2001,99.00000,99.00000,99.00000,EUROPE,0.00000,99.000000,0.000000


## Save Engineered Data

In [10]:
# Saving the processed datasets to feature engineered versions
df_avg.to_csv('../data/implemented_data/avg_years_of_education_p.csv', index=False)
df_ill.to_csv('../data/implemented_data/illeteracy_p.csv', index=False)
df_lit.to_csv('../data/implemented_data/literacy_p.csv', index=False)

print("Feature engineering complete and datasets saved.")


Feature engineering complete and datasets saved.
